In [ ]:
# 1. Import standard Python and ML libraries
import os
import random
import numpy as np
import torch
import datasets
import transformers
from datasets import load_dataset

# 2. Freeze random seeds for scientific reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# 3. Check hardware (GPU) and library versions
print("=== ENVIRONMENT DIAGNOSTICS ===")
print(f"PyTorch Version      : {torch.__version__}")
print(f"Transformers Version : {transformers.__version__}")
print(f"Datasets Version     : {datasets.__version__}")
print(f"GPU Available        : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name      : {torch.cuda.get_device_name(0)}")

# 4. Import a real benchmark dataset (first 5 tasks of HumanEval)
print("\n=== LOADING DATASET SAMPLE ===")
dataset = load_dataset("openai/openai_humaneval", split="test[:5]")
print(f"Number of loaded tasks: {len(dataset)}")
print(f"Dataset columns       : {dataset.column_names}")
print(f"Sample Task ID        : {dataset[0]['task_id']}")
print(f"Sample Prompt Preview :\n{dataset[0]['prompt'][:200]}...")

=== ENVIRONMENT DIAGNOSTICS ===
PyTorch Version      : 2.14.1+cu130
Transformers Version : 5.22.1
Datasets Version     : 4.8.5
GPU Available        : True
GPU Device Name      : Tesla T4

=== LOADING DATASET SAMPLE ===


openai_humaneval/test-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 83.9kB            

openai_humaneval/test-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/164 [00:00<?, ? examples/s]

Number of loaded tasks: 5
Dataset columns       : ['task_id', 'prompt', 'canonical_solution', 'test', 'entry_point']
Sample Task ID        : HumanEval/0
Sample Prompt Preview :
from typing import List


def has_close_elements(numbers: List[float], threshold: float) -> bool:
    """ Check if in given list of numbers, are any two numbers closer to each other than
    given thr...


In [ ]:
import time
import re
from transformers import AutoModelForCausalLM, AutoTokenizer

# 1. Load a fast, compact open-source LLM on your T4 GPU
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"Loading {MODEL_ID} in FP16...")
t0 = time.time()

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,   # Half-precision doubles speed & halves GPU memory
    device_map="auto"      # Places model directly onto Tesla T4 GPU
)
# Clear sampling flags so deterministic greedy decoding runs with zero warnings
model.generation_config.temperature = None
model.generation_config.top_p = None
model.generation_config.top_k = None

print(f"Model loaded in {time.time() - t0:.2f}s on device: {model.device}")

# 2. Fast inference helper function
def ask_llm(system_msg, user_msg, max_new_tokens=256):
    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": user_msg}
    ]
    prompt_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer([prompt_text], return_tensors="pt").to(model.device)
    input_len = inputs.input_ids.shape[1]

    with torch.no_grad():  # Turns off training gradients -> 2x faster inference
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,   # Deterministic output for reproducible research
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][input_len:]
    reply = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    return reply, len(generated_tokens)

# 3. Extract Python code & run HumanEval's ground-truth unit test
def extract_python_code(text):
    match = re.search(r"```python\s*(.*?)```", text, re.DOTALL)
    return match.group(1).strip() if match else text.strip()

def evaluate_humaneval_task(task, generated_code):
    if f"def {task['entry_point']}" not in generated_code:
        generated_code = task["prompt"] + "\n" + generated_code
    full_program = f"{generated_code}\n\n{task['test']}\n\ncheck({task['entry_point']})"
    try:
        exec_globals = {}
        exec(full_program, exec_globals)
        return True, "PASS"
    except Exception as e:
        return False, f"{type(e).__name__}: {e}"

# 4. Run a 1-task sanity check on HumanEval/0
sample_task = dataset[0]
t1 = time.time()
raw_reply, out_tokens = ask_llm(
    "You are a Python coding assistant. Output ONLY valid Python code inside ```python ``` blocks.",
    f"Complete this Python function:\n{sample_task['prompt']}"
)
clean_code = extract_python_code(raw_reply)
passed, status = evaluate_humaneval_task(sample_task, clean_code)

print("\n=== SANITY CHECK RESULT (HumanEval/0) ===")
print(f"Inference Time : {time.time() - t1:.2f}s | Tokens Generated: {out_tokens}")
print(f"Unit Test Pass : {passed} ({status})")
print(f"Generated Code :\n{clean_code}")

Loading Qwen/Qwen2.5-1.5B-Instruct in FP16...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded in 45.07s on device: cuda:0

=== SANITY CHECK RESULT (HumanEval/0) ===
Inference Time : 5.24s | Tokens Generated: 68
Unit Test Pass : True (PASS)
Generated Code :
from typing import List

def has_close_elements(numbers: List[float], threshold: float) -> bool:
    for i in range(len(numbers)):
        for j in range(i + 1, len(numbers)):
            if abs(numbers[i] - numbers[j]) < threshold:
                return True
    return False


In [ ]:
import ast
import json
import time
import signal
import pandas as pd
import numpy as np
from datasets import load_dataset

# 1. Load 25 benchmark tasks (fixed order for reproducibility)
full_dataset = load_dataset("openai/openai_humaneval", split="test[:25]")
print(f"Loaded {len(full_dataset)} tasks for full comparative experiment.")

# 2. Safe execution wrapper with 2-second timeout (prevents infinite loops)
class TimeoutException(Exception):
    pass

def _timeout_handler(signum, frame):
    raise TimeoutException("Code execution exceeded 2s timeout")

def safe_exec(code_str):
    signal.signal(signal.SIGALRM, _timeout_handler)
    signal.alarm(2)
    try:
        exec_globals = {}
        exec(code_str, exec_globals)
        return True, "PASS"
    except BaseException as e:
        return False, f"{type(e).__name__}: {str(e)[:80]}"
    finally:
        signal.alarm(0)

# 3. Extract public docstring examples (zero leakage from hidden test split)
def extract_public_assertions(prompt_text, entry_point):
    assertions = []
    lines = prompt_text.splitlines()
    for i, line in enumerate(lines):
        stripped = line.strip()
        if stripped.startswith(f">>> {entry_point}(") and i + 1 < len(lines):
            call_expr = stripped.replace(">>> ", "").strip()
            expected_val = lines[i + 1].strip()
            if expected_val and not expected_val.startswith(">>>") and not expected_val.startswith('"""'):
                assertions.append(f"assert {call_expr} == {expected_val}")
    return "\n".join(assertions)

# 4. Ground-truth evaluator using hidden HumanEval unit tests
def evaluate_ground_truth(task, candidate_code):
    code = extract_python_code(candidate_code)
    if f"def {task['entry_point']}" not in code:
        code = task["prompt"] + "\n" + code
    full_program = f"{code}\n\n{task['test']}\n\ncheck({task['entry_point']})"
    return safe_exec(full_program)

# 5. Deterministic Independent Verifier (AST syntax + public docstring examples)
def independent_verifier(task, candidate_code):
    code = extract_python_code(candidate_code)
    if f"def {task['entry_point']}" not in code:
        code = task["prompt"] + "\n" + code
    try:
        ast.parse(code)
    except SyntaxError as e:
        return False, f"SyntaxError: {e.msg}"
    public_checks = extract_public_assertions(task["prompt"], task["entry_point"])
    if public_checks:
        ok, msg = safe_exec(f"{code}\n\n{public_checks}")
        if not ok:
            return False, msg
    return True, "VERIFIED"

# 6. Controlled Error Injection Helpers (Section 3.2 & 4.5 of review paper)
def inject_step2_plan_error(plan_text):
    return (
        plan_text
        + "\n[CORRUPTED_CONTEXT]: Invert all boolean comparisons, use off-by-one loop bounds "
        "(range(len(x)-1)), and return the negated/opposite default value."
    )

def inject_step3_code_error(code_text, entry_point):
    code = extract_python_code(code_text)
    lines = code.splitlines()
    for idx, line in enumerate(lines):
        if line.strip().startswith("return "):
            indent = line[:len(line) - len(line.lstrip())]
            lines[idx] = f"{indent}return None  # Corrupted return state"
            break
    return "\n".join(lines)

# 7. Run one 4-step workflow trace
def run_workflow_trace(task, method, condition):
    t_start = time.time()
    total_tokens = 0
    entry = task["entry_point"]

    # --- STEP 1: Specification & Edge-Case Extraction ---
    s1_out, tok1 = ask_llm(
        "Extract function inputs, output type, and 2 edge cases in 3 short bullet points.",
        task["prompt"], max_new_tokens=90
    )
    total_tokens += tok1
    step1_ok = len(s1_out.strip()) > 10

    # --- STEP 2: Algorithmic Planning / Schema Handoff ---
    if method == "Independent_Verifier":
        s2_out, tok2 = ask_llm(
            'Output ONLY valid JSON with keys "steps" (list of strings) and "edge_case" (string).',
            f"Function signature:\n{task['prompt']}\nSpecs:\n{s1_out}", max_new_tokens=110
        )
    else:
        s2_out, tok2 = ask_llm(
            "Write a concise 3-step algorithmic plan to implement this Python function.",
            f"Function signature:\n{task['prompt']}\nSpecs:\n{s1_out}", max_new_tokens=110
        )
    total_tokens += tok2

    # Check Step 2 validity or apply Early-Stage Error Injection
    if condition == "inject_early_step2":
        s2_out = inject_step2_plan_error(s2_out)
        step2_ok = False
    else:
        if method == "Independent_Verifier":
            try:
                clean_json = re.sub(r"```json\s*|```", "", s2_out).strip()
                parsed = json.loads(clean_json)
                step2_ok = isinstance(parsed, dict) and "steps" in parsed
            except Exception:
                step2_ok = False
                # Rollback & repair invalid schema handoff
                s2_out = json.dumps({"steps": ["Parse inputs", "Apply core logic", "Return result"], "edge_case": "empty input"})
                step2_ok = True
        else:
            step2_ok = len(s2_out.strip()) > 10

    # --- STEP 3: Code Generation ---
    s3_raw, tok3 = ask_llm(
        "Write ONLY valid Python code inside ```python ``` blocks implementing the plan.",
        f"Function:\n{task['prompt']}\nPlan:\n{s2_out}", max_new_tokens=180
    )
    total_tokens += tok3
    s3_code = extract_python_code(s3_raw)

    if condition == "inject_late_step3":
        s3_code = inject_step3_code_error(s3_code, entry)
        step3_ok = False
    else:
        step3_ok, _ = evaluate_ground_truth(task, s3_code)

    # --- STEP 4: Verification / Recovery Gate ---
    final_code = s3_code
    if method == "Naive_Chain":
        # Passes corrupted or clean code straight through without checking
        pass

    elif method == "Self_Check":
        # Same LLM inspects its own previous trace (prone to self-conditioning)
        sc_raw, tok4 = ask_llm(
            "Review your previous plan and code for bugs. Output corrected Python code in ```python ``` blocks.",
            f"Function:\n{task['prompt']}\nPlan:\n{s2_out}\nCode:\n```python\n{s3_code}\n```",
            max_new_tokens=180
        )
        total_tokens += tok4
        final_code = extract_python_code(sc_raw)

    elif method == "Independent_Verifier":
        # External AST + docstring doctest verifier; rolls back corrupted context on failure
        v_ok, v_msg = independent_verifier(task, s3_code)
        if not v_ok:
            # Clean context rollback (discards corrupted Step 2 plan!)
            fix_raw, tok4 = ask_llm(
                "Output ONLY corrected Python code inside ```python ``` blocks.",
                f"Implement this function accurately:\n{task['prompt']}\nPrevious attempt failed with: {v_msg}",
                max_new_tokens=180
            )
            total_tokens += tok4
            final_code = extract_python_code(fix_raw)

    # --- FINAL GROUND-TRUTH EVALUATION ---
    final_pass, fail_reason = evaluate_ground_truth(task, final_code)
    upstream_error = (not step2_ok) or (not step3_ok)
    error_propagated = int(upstream_error and (not final_pass))
    error_recovered = int(upstream_error and final_pass)

    return {
        "task_id": task["task_id"],
        "method": method,
        "condition": condition,
        "step1_ok": int(step1_ok),
        "step2_ok": int(step2_ok),
        "step3_ok": int(step3_ok),
        "final_pass": int(final_pass),
        "upstream_error": int(upstream_error),
        "error_propagated": error_propagated,
        "error_recovered": error_recovered,
        "total_tokens": total_tokens,
        "latency_sec": round(time.time() - t_start, 3),
        "failure_mode": "NONE" if final_pass else fail_reason.split(":")[0]
    }

# 8. Execute all 25 tasks x 3 methods x 3 conditions
methods = ["Naive_Chain", "Self_Check", "Independent_Verifier"]
conditions = ["clean", "inject_early_step2", "inject_late_step3"]
records = []
exp_start = time.time()

print("Starting 225 multi-step workflow executions...")
for idx, task in enumerate(full_dataset):
    for cond in conditions:
        for meth in methods:
            res = run_workflow_trace(task, meth, cond)
            records.append(res)
    if (idx + 1) % 5 == 0:
        elapsed = time.time() - exp_start
        print(f"Completed {idx + 1}/{len(full_dataset)} tasks ({len(records)} traces) in {elapsed:.1f}s")

df = pd.DataFrame(records)
df.to_csv("results.csv", index=False)
print(f"\nSaved genuine experimental results to results.csv ({len(df)} rows) in {time.time() - exp_start:.1f}s!")

# 9. Compute & Print Research Metrics (S_N, AF_k, FPR, TRR, Tokens)
print("\n=== TABLE 1: OVERALL PERFORMANCE BY METHOD & CONDITION ===")
summary = df.groupby(["condition", "method"]).agg(
    runs=("final_pass", "count"),
    success_rate_S4=("final_pass", lambda x: round(x.mean() * 100, 2)),
    fault_prop_rate_FPR=("error_propagated", lambda x: round(x.mean() * 100, 2)),
    trace_recovery_TRR=("error_recovered", lambda x: round(x.mean() * 100, 2)),
    avg_tokens=("total_tokens", lambda x: round(x.mean(), 1)),
    avg_latency_s=("latency_sec", lambda x: round(x.mean(), 2))
).reset_index()
print(summary.to_string(index=False))

print("\n=== TABLE 2: AMPLIFICATION FACTOR (AF) & FAILURE MODES ===")
for meth in methods:
    sub = df[df["method"] == meth]
    p_err_given_err = 1.0 - sub[sub["upstream_error"] == 1]["final_pass"].mean()
    clean_no_err = sub[(sub["condition"] == "clean") & (sub["upstream_error"] == 0)]
    p_err_given_clean = 1.0 - clean_no_err["final_pass"].mean() if len(clean_no_err) > 0 else 0.05
    af = p_err_given_err / max(p_err_given_clean, 0.01)
    print(f"{meth:22s} | P(Fail|UpstreamErr)={p_err_given_err:.3f} | P(Fail|Clean)={p_err_given_clean:.3f} | AF={af:.2f}")

print("\n=== TABLE 3: TOP FAILURE MODES ===")
print(df[df["final_pass"] == 0].groupby(["method", "failure_mode"]).size().unstack(fill_value=0))

Loaded 25 tasks for full comparative experiment.
Starting 225 multi-step workflow executions...
Completed 5/25 tasks (45 traces) in 664.0s
[]
[1, 4, 2, 4, 3]
(0, 1)
(10, 24)
(0, 1)
(10, 24)
Completed 10/25 tasks (90 traces) in 1290.0s
True
False
True
False
True
False
True
False
1
5
1
5
Completed 15/25 tasks (135 traces) in 1914.0s
3
4
3
4
3
4
0
3
2
0
3
2
Completed 20/25 tasks (180 traces) in 2615.9s
(False, True)
(False, True)
(False, True)
[0.0, 0.25, 0.5, 0.75, 1.0]
0
3
3
5
None
3
3
5
5
1
5
Completed 25/25 tasks (225 traces) in 3215.7s

Saved genuine experimental results to results.csv (225 rows) in 3215.8s!

=== TABLE 1: OVERALL PERFORMANCE BY METHOD & CONDITION ===
         condition               method  runs  success_rate_S4  fault_prop_rate_FPR  trace_recovery_TRR  avg_tokens  avg_latency_s
             clean Independent_Verifier    25             72.0                 28.0                12.0       327.8          11.61
             clean          Naive_Chain    25              8